<a href="https://colab.research.google.com/github/cermegno/llm-evals/blob/main/02-deepeval-single-turn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeepEval Single-Turn evaluation


In [ ]:
!pip install deepeval langchain langchain-nvidia-ai-endpoints langchain-core --quiet

In [ ]:
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
import json
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

## Nvidia setup

In [ ]:
# Initialize the LLM
from google.colab import userdata
apikey = userdata.get('apikey')
os.environ["NVIDIA_API_KEY"] = apikey
#JUDGE model  ChatNVIDIA(model="openai/gpt-oss-120b")

We need to wrap the Nvidia model

In [ ]:
from deepeval.models import DeepEvalBaseLLM

class CustomNvidiaLLM(DeepEvalBaseLLM):
    model_name = "openai/gpt-oss-120b"

    def __init__(self):
        os.environ["NVIDIA_API_KEY"] = os.environ.get("NVIDIA_API_KEY", "")
        self.model = ChatNVIDIA(model=self.model_name)

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        model = self.load_model()
        # DeepEval expects a plain string back
        return model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        model = self.load_model()
        res = await model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return self.model_name

## Create Dataset
Pre-defined questions. Generate answers from the LLM we want to test

In [ ]:
questions = [
    "What is the capital of Australia",
    "How much is 2 * 24",
    "What are the ingredients of basic bread",
    "Translate 'I am hungry' to spanish",
    "Name the benefits of eating healthy"
]

In [ ]:
llm_to_test = ChatNVIDIA(model="meta/llama-3.1-8b-instruct")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Respond concisely, as simple as possible, in plain text, no markdown."),
    ("user", "{input}")
])

prediction_chain = prompt | llm_to_test

# Query the test model to create the list of answers
answers = []
for question in questions:
    answers.append(prediction_chain.invoke(question).content)

In [ ]:
print(json.dumps(answers, indent=2))

[
  "The capital of Australia is Canberra.",
  "48",
  "The basic ingredients of bread are flour, yeast, water, salt.",
  "Estoy hambriento",
  "Eating healthy provides numerous benefits including:\n\nWeight management\nImproved energy levels\nReduced risk of chronic diseases such as heart disease, diabetes, and certain cancers\nStronger immune system\nBetter digestion and bowel health\nImproved mental health and mood\nIncreased lifespan\nImproved skin and hair health\nReduced risk of osteoporosis\nBetter bone health\nImproved athletic performance\n\nThese benefits can be achieved by consuming a balanced diet rich in fruits, vegetables, whole grains, lean proteins, and healthy fats."
]


## Set up Deepeval
Test case and metrics


In [ ]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input=q,
        actual_output=a,          # model answer
        # no context, no expected_output
    )
    for q, a in zip(questions, answers)
]

In [ ]:
test_cases

[LLMTestCase(input='What is the capital of Australia', actual_output='The capital of Australia is Canberra.', expected_output=None, context=None, retrieval_context=None, metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input='How much is 2 * 24', actual_output='48', expected_output=None, context=None, retrieval_context=None, metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input='What are the ingredients of basic bread', actual_output='The basic ingredients of bread are flour, yeast, water, salt.', expected_output=None

## Evaluate

In [ ]:
from deepeval.evaluate import evaluate
from deepeval.metrics import AnswerRelevancyMetric

metric = AnswerRelevancyMetric(model=nvidia_model, threshold=0.7)

results = evaluate(test_cases=test_cases, metrics=[metric])

In [ ]:
for i, tr in enumerate(results.test_results):
    md = tr.metrics_data[0]
    print(f"Test case {i+1}: {tr.input}")
    print(f"  Score:  {md.score}")
    print(f"  Passed: {md.success}")
    print(f"  Reason: {md.reason}\n")

avg_score = sum(tr.metrics_data[0].score for tr in test_results) / len(test_results)
print(f"Average Answer Relevancy score (NVIDIA eval model): {avg_score:.3f}")

Test case 1: What are the ingredients of basic bread
  Score:  1.0
  Passed: True
  Reason: The score is 1.00 because the response directly listed the basic bread ingredients with no irrelevant statements.

Test case 2: How much is 2 * 24
  Score:  1.0
  Passed: True
  Reason: The score is 1.00 because the answer directly addressed the question with the correct calculation and included no irrelevant statements.

Test case 3: Translate 'I am hungry' to spanish
  Score:  1.0
  Passed: True
  Reason: The score is 1.00 because the response gave the correct Spanish translation and included no irrelevant material.

Test case 4: What is the capital of Australia
  Score:  1.0
  Passed: True
  Reason: The score is 1.00 because the response directly answered the question about Australia's capital with no irrelevant statements.

Test case 5: Name the benefits of eating healthy
  Score:  1.0
  Passed: True
  Reason: The score is 1.00 because the answer directly listed the benefits of eating health